## Model

model.py

### What model.py Handle ?

* Automatically switches models for large datasets
* Selects numerical features
* Scales features
* Trains selected anomaly detection model
* Generates predictions
* Computes anomaly scores
* Adds results to dataframe
* Returns trained model

It is a dynamic, scalable anomaly detection engine.

#### Information :

* This is the core ML engine of your entire Energy AI system.
* The train_model() function dynamically selects and trains an unsupervised anomaly detection algorithm based on dataset size and user preference.
* It ensures scalability by automatically switching to Isolation Forest for large datasets.
* The function standardizes features, generates anomaly predictions and severity scores, and integrates results into the dataset for downstream business analysis.

## Import Models

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope

Model	     -------Type	---------Best For
* Isolation Forest    -------	Tree-based --------	Large datasets
* LOF	          --------Density-based ------	Local anomalies
* One-Class SVM   -------	Boundary-based	 -----Complex shapes
* Robust Covariance  --------	Statistical-------- Gaussian data

This makes our system flexible.

## Large Dataset Auto-Switching

In [ ]:
def train_model(df, model_choice="Isolation Forest", large_data_threshold=200000):

If dataset > 200,000 rows:

It forces:

model_choice = "Isolation Forest"

Why?

Because:

LOF → O(n²) complexity

OneClassSVM → very slow on big data

EllipticEnvelope → covariance expensive

IsolationForest → O(n log n) → fast & scalable

This makes our system production-ready.


## Feature Selection

In [ ]:
X = df.select_dtypes(include=np.number).drop(columns=["final_anomaly"], errors="ignore")

* Only numerical features
* Removes target column if exists

This ensures:

Model only trains on valid numeric data.

## Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

StandardScaler:

Transforms data into:

Mean = 0
Std = 1

Why important?

Models like:

SVM

LOF

Covariance

are distance-based.

If one feature has large values (e.g., 10000) and another small (0.2):

Model becomes biased.

Scaling fixes that.

## Model Selection Logic

In [ ]:
model = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)


contamination=0.05 → assume 5% anomalies

* Isolation Forest works by:

Randomly splitting data using trees.
Anomalies get isolated faster → shorter path length.

Then:

preds = model.predict(X_scaled)
scores = model.decision_function(X_scaled)

predict():

-1 → anomaly

1 → normal

decision_function():
Lower value → more anomalous.

*  Local Outlier Factor (LOF)

LOF measures:

How isolated a point is compared to neighbors.

If density of point << density of neighbors
→ anomaly

*  One-Class SVM

Learns a boundary around normal data.

Points outside boundary → anomaly.

Works well for complex nonlinear patterns.

* Robust Covariance (EllipticEnvelope)

Assumes data follows Gaussian distribution.

Fits ellipse around data.

Points outside ellipse → anomaly.

## Converting Predictions

In [ ]:
df["final_anomaly"] = np.where(preds == -1, 1, 0)

Model outputs:

-1 → anomaly

1 → normal

You convert to:

1 → anomaly

0 → normal

Much cleaner for business reporting.

## Anomaly Score

In [ ]:
df["anomaly_score"] = scores

Important:

Lower score = more anomalous.

This allows:

* Ranking anomalies

* Finding top 10 worst cases

* Generating PDF reports

Risk prioritizatio

## Anomaly Rate

In [ ]:
anomaly_rate = df["final_anomaly"].mean() * 100

If:
5% of rows are 1
Then anomaly rate = 5%

Used for:

* Severity classification

* Business reporting

* Executive dashboards

## Return

In [ ]:
return df, switched, model

So our app knows:

* Updated dataframe

* Whether model switched

* Trained model object

## Whole Code 

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import StandardScaler
import numpy as np


def train_model(df, model_choice="Isolation Forest", large_data_threshold=200000):

    print(f"🤖 Training {model_choice}...")

    # =========================
    # Detect Large Dataset
    # =========================
    switched = False

    if len(df) > large_data_threshold and model_choice != "Isolation Forest":
        print("⚠ Large dataset detected. Switching to Isolation Forest.")
        model_choice = "Isolation Forest"
        switched = True

    # =========================
    # Feature Selection
    # =========================
    X = df.select_dtypes(include=np.number).drop(
        columns=["final_anomaly"], errors="ignore"
    )

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # =========================
    # Model Selection
    # =========================
    if model_choice == "Isolation Forest":
        model = IsolationForest(
            contamination=0.05,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_scaled)
        preds = model.predict(X_scaled)
        scores = model.decision_function(X_scaled)

    elif model_choice == "Local Outlier Factor (LOF)":
        model = LocalOutlierFactor(contamination=0.05)
        preds = model.fit_predict(X_scaled)
        scores = model.negative_outlier_factor_

    elif model_choice == "One-Class SVM":
        model = OneClassSVM(nu=0.05)
        model.fit(X_scaled)
        preds = model.predict(X_scaled)
        scores = model.decision_function(X_scaled)

    elif model_choice == "Robust Covariance":
        model = EllipticEnvelope(contamination=0.05)
        model.fit(X_scaled)
        preds = model.predict(X_scaled)
        scores = model.decision_function(X_scaled)

    else:
        raise ValueError("Invalid model selected")

    # =========================
    # Predictions
    # =========================
    df["final_anomaly"] = np.where(preds == -1, 1, 0)

    # Lower score = more anomalous
    df["anomaly_score"] = scores

    anomaly_rate = df["final_anomaly"].mean() * 100

    print("✅ Model Training Completed")
    print(f"🔎 Anomaly Rate: {round(anomaly_rate, 2)}%")

    return df, switched, model